# 02 - Czyszczenie danych

## 1. Import i wczytanie danych

In [ ]:
import pandas as pd

In [ ]:
measurements = pd.read_csv('../data/raw/Measurements.csv') 
travel_time_a = pd.read_csv('../data/raw/Travel_Time_A.csv')
travel_time_b = pd.read_csv('../data/raw/Travel_Time_B.csv')
weather = pd.read_csv('../data/raw/Weather_Conditions.csv')
route_details = pd.read_csv('../data/raw/Route_Details.csv')

## 2. Konwersja typów

In [ ]:
measurements['Date'] = pd.to_datetime(measurements['Date'], format='%Y-%m-%d')

In [ ]:
print(measurements.info())

<class 'pandas.DataFrame'>
RangeIndex: 44588 entries, 0 to 44587
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Measurement_Id    44588 non-null  int64         
 1   Route_Id          44588 non-null  int64         
 2   Direction         44588 non-null  str           
 3   Date              44588 non-null  datetime64[us]
 4   Day_Of_Week       44588 non-null  str           
 5   Measurement_Time  44588 non-null  str           
dtypes: datetime64[us](1), int64(2), str(3)
memory usage: 2.0 MB
None


## 3. Obsługa brakujących wartości w Travel_Time

In [ ]:
travel_time = pd.concat([travel_time_a, travel_time_b], ignore_index=True)
travel_time = travel_time.dropna(subset=['Duration_Min'])
travel_time = travel_time.sort_values(by='Measurement_Id').reset_index(drop=True)

print("Ilość zduplikowanych wierszy:", travel_time.duplicated(subset='Measurement_Id').sum())
print("\nInformacje o danych:")
print(travel_time.info())
print("\nPierwsze 10 wierszy danych:")
print(travel_time.head(10))

Ilość zduplikowanych wierszy: 0

Informacje o danych:
<class 'pandas.DataFrame'>
RangeIndex: 44588 entries, 0 to 44587
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Measurement_Id  44588 non-null  int64  
 1   Duration_Min    44588 non-null  float64
 2   Delay_Min       44588 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 1.0 MB
None

Pierwsze 10 wierszy danych:
   Measurement_Id  Duration_Min  Delay_Min
0               1          42.0        2.0
1               2          39.0        2.0
2               3          39.0        4.0
3               4          43.0        3.0
4               5          44.0        4.0
5               6          39.0        2.0
6               7          33.0        1.0
7               8          37.0        3.0
8               9          41.0        3.0
9              10          43.0        4.0


In [ ]:
print("Sprawdzam czy ilość wierszy w travel_time pokrywa się z ilością wierszy w measurements:")
print("Ilość wierszy w travel_time:", len(travel_time))
print("Ilość wierszy w measurements:", len(measurements))

Sprawdzam czy ilość wierszy w travel_time pokrywa się z ilością wierszy w measurements:
Ilość wierszy w travel_time: 44588
Ilość wierszy w measurements: 44588


## 4. Anomalie w Duration_Min

In [ ]:
print("Średnia wartość Duration_Min:", travel_time['Duration_Min'].mean().round(2))
print("Średnia wartość Delay_Min:", travel_time['Delay_Min'].mean().round(2))
print("Percentyle 1%, 50% i 99% dla Duration_Min:")
travel_time.quantile([0.01, 0.5, 0.99])

Średnia wartość Duration_Min: 43.61
Średnia wartość Delay_Min: 6.41
Percentyle 1%, 50% i 99% dla Duration_Min:


,Measurement_Id,Duration_Min,Delay_Min
0.01,446.87,32.0,0.0
0.50,22296.50,43.0,5.0
0.99,44147.13,61.0,25.0


In [ ]:
print("Minimalna wartość Duration_Min:", travel_time['Duration_Min'].min())
print("Maksymalna wartość Duration_Min:", travel_time['Duration_Min'].max())

print("Ilość wierszy z Duration_Min > 80:", travel_time[travel_time['Duration_Min'] > 80]['Duration_Min'].count())


Minimalna wartość Duration_Min: 32.0
Maksymalna wartość Duration_Min: 86.0
Ilość wierszy z Duration_Min > 80: 2


Zbiór wydaje się nie mieć specyficznych, odstających wartości. Znalazłem dwa rekordy powyżej 80 minut, co jest realistyczne przy tym zbiorze, szczególnie biorąc pod uwagę najdłuższe trasy i godziny szczytu.

## 5. Złożenie głównego DataFrame

In [ ]:
df = pd.merge(measurements, travel_time, on='Measurement_Id', how='inner')
weather = weather.drop(columns=['Snow_Origin', 'Snow_Destination'])
df = pd.merge(df, weather, on='Measurement_Id', how='left')
print("Ilość wierszy w połączonym DataFrame:", len(df))

Ilość wierszy w połączonym DataFrame: 44588


In [ ]:
print(df.shape)
print(df.info())

df.describe().round(2)

(44588, 18)
<class 'pandas.DataFrame'>
RangeIndex: 44588 entries, 0 to 44587
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Measurement_Id           44588 non-null  int64         
 1   Route_Id                 44588 non-null  int64         
 2   Direction                44588 non-null  str           
 3   Date                     44588 non-null  datetime64[us]
 4   Day_Of_Week              44588 non-null  str           
 5   Measurement_Time         44588 non-null  str           
 6   Duration_Min             44588 non-null  float64       
 7   Delay_Min                44588 non-null  float64       
 8   Weather_Origin           44588 non-null  str           
 9   Temperature_Origin       44588 non-null  float64       
 10  Humidity_Origin          44588 non-null  int64         
 11  Rain_Origin              44588 non-null  float64       
 12  Visibility_Origin        44588 

,Measurement_Id,Route_Id,Date,Duration_Min,Delay_Min,Temperature_Origin,Humidity_Origin,Rain_Origin,Visibility_Origin,Temperature_Destination,Humidity_Destination,Rain_Destination,Visibility_Destination
count,44588.00,44588.00,44588,44588.00,44588.00,44588.00,44588.00,44588.00,44588.00,44588.00,44588.00,44588.00,44588.00
mean,22295.91,3.00,2025-07-15 23:37:00.328339,43.61,6.41,19.48,76.26,0.35,9768.26,19.48,76.25,0.35,9766.96
min,1.00,1.00,2025-07-01 00:00:00,32.00,0.00,10.00,23.00,0.00,150.00,10.00,23.00,0.00,150.00
25%,11147.75,2.00,2025-07-08 00:00:00,40.00,2.00,15.99,63.00,0.00,10000.00,15.99,63.00,0.00,10000.00
50%,22296.50,3.00,2025-07-16 00:00:00,43.00,5.00,18.77,79.00,0.00,10000.00,18.77,79.00,0.00,10000.00
75%,33443.25,4.00,2025-07-24 00:00:00,47.00,10.00,22.49,92.00,0.00,10000.00,22.49,92.00,0.00,10000.00
max,44593.00,5.00,2025-07-31 00:00:00,86.00,47.00,33.28,100.00,42.11,10000.00,33.28,100.00,42.11,10000.00
std,12872.66,1.41,NaN,6.07,5.47,4.60,18.01,1.59,1080.76,4.60,18.02,1.57,1084.79


## 6. Zapis do pliku

In [ ]:
df.to_csv('../data/processed/traveltime_clean.csv', index=False)